# Revision 03 Four-Condition Ratio-Matched Paper Figures


Figure notebook for the targeted paragraph-scrambled validation.

It generates:

1. **reviewer benchmark figure:** raw HTFA + conventional PCA + spatial AA + temporal AA
   for all four conditions;
2. the same comparison as **fraction of above-chance raw HTFA information retained**,
   which makes conditions with different \(T\) directly interpretable;
3. **paper Figure 3-style** full spatial and temporal AA curves with paragraph added;
4. **paper Figure 4-style** top-5/top-10 spatial-vs-temporal matched-ratio curves with
   paragraph added;
5. **paper Figure 5-style** spatial-minus-temporal top-\(m\) heatmaps for all four conditions;
6. a compact condition hierarchy plot at selected representational ratios.

Because these are within-condition decompositions, this notebook should be presented as a
four-condition validation/sensitivity analysis rather than as a replacement for the original
across-condition shared-basis results.


## Paragraph parsing note

This figure notebook assumes notebooks 019 and 020 were run with the duplicated paragraph entry removed. It reads only the saved summary tables, so no additional subject filtering is required here.

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

DECODE_DIR = Path("four_condition_ratio_decoding_outputs")

RAW_CSV = DECODE_DIR / "raw_htfa_summary.csv"
PCA_CSV = DECODE_DIR / "pca_summary_with_retention.csv"
AA_FULL_CSV = DECODE_DIR / "aa_full_summary_with_retention.csv"
AA_TOPM_CSV = DECODE_DIR / "aa_topm_summary_with_retention.csv"

CONDITIONS = ["intact", "paragraph", "word", "rest"]

COND_LABELS = {
    "intact": "Intact",
    "paragraph": "Paragraph-scrambled",
    "word": "Word-scrambled",
    "rest": "Rest",
}

COND_COLORS = {
    "intact": "purple",
    "paragraph": "orange",
    "word": "green",
    "rest": "black",
}

ANALYSIS_LINESTYLES = {
    "spatial": "-",
    "temporal": "--",
}

ANALYSIS_MARKERS = {
    "spatial": "o",
    "temporal": "s",
}

TOP_M_FOR_RATIO_FIGURES = [5, 10]
HEATMAP_TOP_M = [1, 2, 3, 5, 7, 10, 15]

# Fixed ratios for compact hierarchy summary.
SUMMARY_RATIOS = [0.05, 0.10, 0.20]

FIG_ROOT = Path("outputs/notebook_figures")
FIG_DIR = FIG_ROOT / "021_four_condition_ratio_validation"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

In [ ]:
# ============================================================
# IMPORTS / LOAD DATA
# ============================================================

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

raw = pd.read_csv(RAW_CSV)
pca = pd.read_csv(PCA_CSV)
aa_full = pd.read_csv(AA_FULL_CSV)

topm = (
    pd.read_csv(AA_TOPM_CSV)
    if AA_TOPM_CSV.exists()
    else pd.DataFrame()
)

_fig_counter = 0

def save_current_fig(name):
    global _fig_counter

    if not SAVE_FIGS:
        return None

    _fig_counter += 1

    safe = "".join(
        c if c.isalnum() or c in "_-."
        else "_"
        for c in str(name)
    )

    out = (
        FIG_DIR
        / f"{_fig_counter:03d}_{safe}.{FIG_FORMAT}"
    )

    plt.savefig(
        out,
        dpi=DPI,
        bbox_inches="tight",
    )

    print("Saved:", out)

    return out

print("raw rows:", len(raw))
print("PCA rows:", len(pca))
print("AA full rows:", len(aa_full))
print("AA top-m rows:", len(topm))

## Figure A: PCA + raw HTFA + spatial AA + temporal AA

Each panel is one condition. The raw HTFA baseline is shown as a horizontal reference.
PCA and spatial AA use their node-space ratio \(K/700\); temporal AA uses \(K/T_c\).

Because absolute chance differs when \(T_c\) differs, this figure should always be interpreted
alongside the normalized retention figure below.

In [ ]:
# ============================================================
# FIGURE A — ABSOLUTE ACCURACY
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.0, 8.0),
    sharex=True,
)

axes = axes.ravel()

for ax, condition in zip(
    axes,
    CONDITIONS,
):

    # Raw
    r = raw[
        raw["condition"] == condition
    ].iloc[0]

    ax.axhline(
        r["mean"],
        color="gray",
        linestyle=":",
        linewidth=2,
        label="Raw HTFA",
    )

    ax.axhline(
        r["chance"],
        color="lightgray",
        linestyle="--",
        linewidth=1.2,
        label="Chance",
    )

    # PCA
    p = pca[
        pca["condition"] == condition
    ].sort_values("actual_ratio")

    ax.errorbar(
        p["actual_ratio"],
        p["mean"],
        yerr=p["err"],
        linestyle="-.",
        marker="^",
        markersize=4,
        capsize=2,
        linewidth=1.8,
        label="PCA",
    )

    # AA
    for analysis_type in [
        "spatial",
        "temporal",
    ]:

        a = aa_full[
            (aa_full["condition"] == condition)
            & (aa_full["analysis_type"] == analysis_type)
        ].sort_values("actual_ratio")

        ax.errorbar(
            a["actual_ratio"],
            a["mean"],
            yerr=a["err"],
            linestyle=ANALYSIS_LINESTYLES[
                analysis_type
            ],
            marker=ANALYSIS_MARKERS[
                analysis_type
            ],
            markersize=4,
            capsize=2,
            linewidth=2.0,
            label=f"{analysis_type.capitalize()} AA",
        )

    ax.set_title(
        COND_LABELS[condition]
    )

    ax.set_xlabel(
        "Normalized component ratio"
    )

    ax.set_ylabel(
        "Timepoint-identification accuracy"
    )

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    frameon=False,
    loc="upper center",
    ncol=5,
)

fig.suptitle(
    "Four-condition information preservation: raw HTFA, PCA, and MS-AA",
    y=1.02,
)

plt.tight_layout()

save_current_fig(
    "four_condition_raw_PCA_spatial_temporal_absolute"
)

plt.show()
plt.close()

## Figure B: fraction of raw above-chance information retained

For condition \(c\),

\[
R_c =
\frac{A_{\mathrm{method},c}-1/T_c}
     {A_{\mathrm{raw},c}-1/T_c}.
\]

Thus \(R=1\) means the reduced representation preserves all above-chance
timepoint-identifiable information available in the raw 700-node HTFA representation.
This is the cleanest four-condition comparison when temporal lengths differ.

In [ ]:
# ============================================================
# FIGURE B — RAW-NORMALIZED RETENTION
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.0, 8.0),
    sharex=True,
)

axes = axes.ravel()

for ax, condition in zip(
    axes,
    CONDITIONS,
):

    ax.axhline(
        1.0,
        color="gray",
        linestyle=":",
        linewidth=1.8,
        label="Raw HTFA",
    )

    p = pca[
        pca["condition"] == condition
    ].sort_values("actual_ratio")

    ax.plot(
        p["actual_ratio"],
        p[
            "fraction_raw_above_chance_retained"
        ],
        linestyle="-.",
        marker="^",
        markersize=4,
        linewidth=1.8,
        label="PCA",
    )

    for analysis_type in [
        "spatial",
        "temporal",
    ]:

        a = aa_full[
            (aa_full["condition"] == condition)
            & (aa_full["analysis_type"] == analysis_type)
        ].sort_values("actual_ratio")

        ax.plot(
            a["actual_ratio"],
            a[
                "fraction_raw_above_chance_retained"
            ],
            linestyle=ANALYSIS_LINESTYLES[
                analysis_type
            ],
            marker=ANALYSIS_MARKERS[
                analysis_type
            ],
            markersize=4,
            linewidth=2.0,
            label=f"{analysis_type.capitalize()} AA",
        )

    ax.set_title(
        COND_LABELS[condition]
    )

    ax.set_xlabel(
        "Normalized component ratio"
    )

    ax.set_ylabel(
        "Fraction of raw above-chance\ninformation retained"
    )

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    frameon=False,
    loc="upper center",
    ncol=4,
)

fig.suptitle(
    "Compression of stimulus-locked structure across four conditions",
    y=1.02,
)

plt.tight_layout()

save_current_fig(
    "four_condition_fraction_raw_retained"
)

plt.show()
plt.close()

## Figure C: paper Figure 3-style full MS-AA curves

This is the closest four-condition analogue of the current manuscript's full-reconstruction
panels, but the x-axis is normalized ratio because paragraph has a different temporal
dimensionality.

In [ ]:
# ============================================================
# FIGURE C — PAPER FIGURE 3 STYLE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.6),
    sharey=True,
)

for ax, analysis_type in zip(
    axes,
    ["spatial", "temporal"],
):

    for condition in CONDITIONS:

        s = aa_full[
            (aa_full["analysis_type"] == analysis_type)
            & (aa_full["condition"] == condition)
        ].sort_values("actual_ratio")

        ax.errorbar(
            s["actual_ratio"],
            s["mean"],
            yerr=s["err"],
            color=COND_COLORS[condition],
            linestyle="-",
            marker="o",
            markersize=3,
            capsize=2,
            linewidth=1.8,
            label=COND_LABELS[condition],
        )

    ax.set_title(
        f"{analysis_type.capitalize()} AA"
    )

    ax.set_xlabel(
        "Normalized component ratio"
    )

    ax.set_ylabel(
        "Timepoint-identification accuracy"
    )

axes[1].legend(
    frameon=False
)

plt.tight_layout()

save_current_fig(
    "paper_Fig3_four_condition_full_ratio"
)

plt.show()
plt.close()

## Figure D: paper Figure 4-style top-5 and top-10 matched-ratio curves

Solid = spatial AA; dashed = temporal AA. Color = condition.

These panels require notebook 020's top-\(m\) section. They reproduce the current
paper figure structure while adding paragraph-scrambled.

In [ ]:
# ============================================================
# FIGURE D — PAPER FIGURE 4 STYLE
# ============================================================

if len(topm) == 0:
    print(
        "No top-m CSV found. Run RUN_AA_TOPM=True in notebook 020."
    )
else:

    fig, axes = plt.subplots(
        1,
        len(TOP_M_FOR_RATIO_FIGURES),
        figsize=(12.0, 4.8),
        sharey=True,
    )

    if len(TOP_M_FOR_RATIO_FIGURES) == 1:
        axes = [axes]

    for ax, m in zip(
        axes,
        TOP_M_FOR_RATIO_FIGURES,
    ):

        for condition in CONDITIONS:

            for analysis_type in [
                "spatial",
                "temporal",
            ]:

                s = topm[
                    (topm["top_m"] == m)
                    & (topm["condition"] == condition)
                    & (
                        topm["analysis_type"]
                        == analysis_type
                    )
                ].sort_values("actual_ratio")

                ax.errorbar(
                    s["actual_ratio"],
                    s["mean"],
                    yerr=s["err"],
                    color=COND_COLORS[
                        condition
                    ],
                    linestyle=ANALYSIS_LINESTYLES[
                        analysis_type
                    ],
                    marker=None,
                    capsize=2,
                    linewidth=2.0,
                )

        ax.set_title(
            f"Top-{m}"
        )

        ax.set_xlabel(
            "Normalized component ratio"
        )

        ax.set_ylabel(
            "Timepoint-identification accuracy"
        )

    condition_handles = [
        Line2D(
            [],
            [],
            color=COND_COLORS[c],
            linewidth=2,
            label=COND_LABELS[c],
        )
        for c in CONDITIONS
    ]

    analysis_handles = [
        Line2D(
            [],
            [],
            color="gray",
            linestyle=ANALYSIS_LINESTYLES[a],
            linewidth=2,
            label=f"{a.capitalize()} AA",
        )
        for a in ["spatial", "temporal"]
    ]

    fig.legend(
        handles=condition_handles + analysis_handles,
        frameon=False,
        loc="upper center",
        ncol=6,
        bbox_to_anchor=(0.5, 1.08),
    )

    plt.tight_layout()

    save_current_fig(
        "paper_Fig4_four_condition_topm_ratio"
    )

    plt.show()
    plt.close()

## Figure E: paper Figure 5-style spatial-minus-temporal heatmaps

Because notebook 019 intentionally creates a common *target ratio grid*, spatial and temporal
values can be compared directly by `target_ratio`; there is no need for nearest-neighbor
matching after the fact.

Positive values mean spatial AA has higher top-\(m\) accuracy.

In [ ]:
# ============================================================
# FIGURE E — SPATIAL MINUS TEMPORAL HEATMAPS
# ============================================================

if len(topm) == 0:
    print(
        "No top-m CSV found. Run RUN_AA_TOPM=True in notebook 020."
    )
else:

    heat_rows = []

    for condition in CONDITIONS:

        for m in HEATMAP_TOP_M:

            s = topm[
                (topm["condition"] == condition)
                & (topm["top_m"] == m)
                & (
                    topm["analysis_type"]
                    == "spatial"
                )
            ]

            t = topm[
                (topm["condition"] == condition)
                & (topm["top_m"] == m)
                & (
                    topm["analysis_type"]
                    == "temporal"
                )
            ]

            merged = s.merge(
                t,
                on=[
                    "condition",
                    "target_ratio",
                    "top_m",
                ],
                suffixes=(
                    "_spatial",
                    "_temporal",
                ),
            )

            for _, r in merged.iterrows():

                heat_rows.append({
                    "condition": condition,
                    "top_m": m,
                    "target_ratio": float(
                        r["target_ratio"]
                    ),
                    "spatial_minus_temporal": float(
                        r["mean_spatial"]
                        - r["mean_temporal"]
                    ),
                })

    heat_df = pd.DataFrame(
        heat_rows
    )

    vmax = np.nanmax(
        np.abs(
            heat_df[
                "spatial_minus_temporal"
            ].to_numpy()
        )
    )

    fig, axes = plt.subplots(
        len(CONDITIONS),
        1,
        figsize=(11.0, 10.0),
        sharex=True,
    )

    for ax, condition in zip(
        axes,
        CONDITIONS,
    ):

        s = heat_df[
            heat_df["condition"]
            == condition
        ]

        piv = s.pivot_table(
            index="top_m",
            columns="target_ratio",
            values="spatial_minus_temporal",
            aggfunc="mean",
        )

        piv = piv.reindex(
            HEATMAP_TOP_M
        )

        piv = piv.reindex(
            sorted(piv.columns),
            axis=1,
        )

        im = ax.imshow(
            piv.values,
            aspect="auto",
            cmap="coolwarm",
            vmin=-vmax,
            vmax=vmax,
        )

        ax.set_yticks(
            np.arange(
                len(piv.index)
            )
        )

        ax.set_yticklabels(
            [str(x) for x in piv.index]
        )

        ax.set_ylabel(
            "top-m"
        )

        ax.set_title(
            COND_LABELS[condition]
        )

    final_piv = piv

    axes[-1].set_xticks(
        np.arange(
            len(final_piv.columns)
        )
    )

    axes[-1].set_xticklabels(
        [
            f"{x:.3g}"
            for x in final_piv.columns
        ],
        rotation=45,
        ha="right",
    )

    axes[-1].set_xlabel(
        "Matched normalized component ratio"
    )

    cbar = fig.colorbar(
        im,
        ax=axes,
        shrink=0.75,
        pad=0.02,
    )

    cbar.set_label(
        "Spatial minus temporal accuracy"
    )

    fig.suptitle(
        "Spatial minus temporal sparse timepoint identification",
        y=0.995,
    )

    plt.tight_layout()

    save_current_fig(
        "paper_Fig5_four_condition_spatial_minus_temporal_heatmap"
    )

    plt.show()
    plt.close()

## Figure F: compact four-condition hierarchy at fixed ratios

This is the figure I would use to answer the paragraph reviewer most directly.

At each selected representational ratio, it shows the fraction of raw above-chance HTFA
information retained by PCA, spatial AA, and temporal AA. If paragraph consistently falls
between intact and word-scrambled under compression, that provides a much stronger graded
validation than the original intact > word > rest comparison.

In [ ]:
# ============================================================
# FIGURE F — CONDITION HIERARCHY AT FIXED RATIOS
# ============================================================

def nearest_row(df, condition, target_ratio):
    s = df[
        df["condition"] == condition
    ].copy()

    if len(s) == 0:
        return None

    idx = (
        s["actual_ratio"]
        - target_ratio
    ).abs().idxmin()

    return s.loc[idx]

methods = [
    ("PCA", None),
    ("Spatial AA", "spatial"),
    ("Temporal AA", "temporal"),
]

fig, axes = plt.subplots(
    1,
    len(SUMMARY_RATIOS),
    figsize=(13.5, 4.3),
    sharey=True,
)

if len(SUMMARY_RATIOS) == 1:
    axes = [axes]

x = np.arange(
    len(CONDITIONS)
)

for ax, ratio in zip(
    axes,
    SUMMARY_RATIOS,
):

    offsets = np.linspace(
        -0.20,
        0.20,
        len(methods),
    )

    for offset, (
        method_label,
        analysis_type,
    ) in zip(
        offsets,
        methods,
    ):

        ys = []

        for condition in CONDITIONS:

            if method_label == "PCA":
                row = nearest_row(
                    pca,
                    condition,
                    ratio,
                )

            else:
                row = nearest_row(
                    aa_full[
                        aa_full["analysis_type"]
                        == analysis_type
                    ],
                    condition,
                    ratio,
                )

            ys.append(
                np.nan
                if row is None
                else row[
                    "fraction_raw_above_chance_retained"
                ]
            )

        ax.plot(
            x + offset,
            ys,
            marker="o",
            linestyle="none",
            markersize=8,
            label=method_label,
        )

    ax.axhline(
        1.0,
        color="gray",
        linestyle=":",
        linewidth=1,
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            COND_LABELS[c]
            for c in CONDITIONS
        ],
        rotation=25,
        ha="right",
    )

    ax.set_title(
        f"ratio ≈ {ratio:g}"
    )

    ax.set_ylabel(
        "Fraction raw above-chance\ninformation retained"
    )

axes[-1].legend(
    frameon=False,
    bbox_to_anchor=(1.04, 1),
    loc="upper left",
)

plt.tight_layout()

save_current_fig(
    "four_condition_hierarchy_fixed_ratios"
)

plt.show()
plt.close()

In [ ]:
# ============================================================
# EXPORT PLOTTING TABLES
# ============================================================

raw.to_csv(
    FIG_DIR / "plotdata_raw.csv",
    index=False,
)

pca.to_csv(
    FIG_DIR / "plotdata_pca.csv",
    index=False,
)

aa_full.to_csv(
    FIG_DIR / "plotdata_aa_full.csv",
    index=False,
)

if len(topm):
    topm.to_csv(
        FIG_DIR / "plotdata_aa_topm.csv",
        index=False,
    )

print("Figures/data saved to:", FIG_DIR)